# 1D-CNN v6 — Expanded Training + 80 Epochs + Better Calibration

Changes: all sessions from non-interday train subjects, 80 epochs, mixup+noise, LogReg C=0.1.

In [1]:
import sys, math
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.linear_model import LogisticRegression

from config import RANDOM_SEED, N_CLASSES, MODELS_DIR, get_device
from src.experiment_runner import (
    get_splits, load_and_norm,
    run_zero_shot, run_calibration, print_comparison,
)
from src.evaluation import measure_latency, print_latency

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
DEVICE = get_device()
splits = get_splits()
print(f'Device: {DEVICE}')

Device: mps


In [2]:
class ECA1d(nn.Module):
    def __init__(self,ch):
        super().__init__()
        k = max(int(abs(math.log2(ch)/2+0.5)),3); k = k if k%2 else k+1
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.conv = nn.Conv1d(1,1,k,padding=k//2,bias=False)
    def forward(self,x):
        b,c,t = x.size()
        return x * torch.sigmoid(self.conv(self.gap(x).view(b,1,c))).view(b,c,1).expand_as(x)

class SepConv1d(nn.Module):
    def __init__(self,ic,oc,k=5,p=2):
        super().__init__()
        self.dw = nn.Conv1d(ic,ic,k,padding=p,groups=ic)
        self.pw = nn.Conv1d(ic,oc,1)
    def forward(self,x): return self.pw(self.dw(x))

class TemporalSCNN(nn.Module):
    def __init__(self,in_ch=8,n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_ch,64,5,padding=2), nn.BatchNorm1d(64), nn.ReLU(), ECA1d(64), nn.MaxPool1d(2),
            SepConv1d(64,128,5,2), nn.BatchNorm1d(128), nn.ReLU(), ECA1d(128), nn.MaxPool1d(2),
            SepConv1d(128,256,3,1), nn.BatchNorm1d(256), nn.ReLU(), ECA1d(256), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(256,64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64,n_classes),
        )
    def forward(self,x): return self.classifier(self.features(x))
    def extract_feat(self,x):
        with torch.no_grad(): return nn.Flatten()(self.features(x))

print(f'Params: {sum(p.numel() for p in TemporalSCNN().parameters()):,}')

Params: 62,676


In [3]:
class MixupDataset(Dataset):
    def __init__(self,X,y,alpha=0.2,noise_std=0.1):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.alpha = alpha; self.noise_std = noise_std; self.n = len(y)
    def __len__(self): return self.n
    def __getitem__(self,i):
        x,y1 = self.X[i].clone(), self.y[i]
        x = x + torch.randn_like(x)*self.noise_std
        if self.alpha>0 and torch.rand(1).item()<0.5:
            j = torch.randint(0,self.n,(1,)).item()
            lam = np.random.beta(self.alpha,self.alpha)
            x = lam*x + (1-lam)*(self.X[j]+torch.randn_like(self.X[j])*self.noise_std)
            y1 = y1 if lam>=0.5 else self.y[j]
        return x, y1

In [4]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Train: {X_train.shape}')

model = TemporalSCNN().to(DEVICE)
loader = DataLoader(MixupDataset(X_train, y_train), batch_size=256, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=80, steps_per_epoch=len(loader))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

for ep in range(80):
    model.train()
    ls,c,t = 0,0,0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE),yb.to(DEVICE)
        out = model(xb); loss = crit(out,yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        ls += loss.item()*xb.size(0); c += (out.argmax(1)==yb).sum().item(); t += xb.size(0)
    if (ep+1)%10==0 or ep==0:
        print(f'Epoch {ep+1:2d}/80 — loss:{ls/t:.4f} acc:{c/t:.4f}')

torch.save(model.state_dict(), MODELS_DIR / '1dcnn_v6.pt')
print('Saved.')

Loading windows: 100%|██████████| 9021/9021 [00:04<00:00, 2196.99it/s]


Train: (1030712, 8, 50)
Epoch  1/80 — loss:1.7447 acc:0.3243
Epoch 10/80 — loss:1.1652 acc:0.6693
Epoch 20/80 — loss:1.1256 acc:0.6906
Epoch 30/80 — loss:1.1049 acc:0.7029
Epoch 40/80 — loss:1.0787 acc:0.7164
Epoch 50/80 — loss:1.0476 acc:0.7324
Epoch 60/80 — loss:1.0115 acc:0.7504
Epoch 70/80 — loss:0.9739 acc:0.7694
Epoch 80/80 — loss:0.9566 acc:0.7784
Saved.


In [5]:
@torch.no_grad()
def cnn1d_predict(X):
    model.eval()
    Xt = torch.from_numpy(X).float()
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model(xb[0].to(DEVICE)).argmax(1).cpu().numpy() for xb in loader])

@torch.no_grad()
def cnn1d_features(X):
    model.eval()
    Xt = torch.from_numpy(X).float()
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model.extract_feat(xb[0].to(DEVICE)).cpu().numpy() for xb in loader])

def cnn1d_finetune(X_cal, y_cal):
    F = cnn1d_features(X_cal)
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=0.1)
    clf.fit(F, y_cal)
    def predict_ft(X): return clf.predict(cnn1d_features(X))
    return predict_ft

print('Option B — Zero-shot:')
zero_results = run_zero_shot(cnn1d_predict, splits, norm_stats)
print('\nOption A — Calibration:')
cal_results = run_calibration(cnn1d_predict, cnn1d_finetune, splits, norm_stats)

Option B — Zero-shot:
  S1 zero-shot: 0.5862
  S2 zero-shot: 0.5352
  S3 zero-shot: 0.5398
  S4 zero-shot: 0.6556
  S5 zero-shot: 0.8029

Option A — Calibration:
  S1 calibrated: 0.7046
  S2 calibrated: 0.7657
  S3 calibrated: 0.7675
  S4 calibrated: 0.8137
  S5 calibrated: 0.8623


In [6]:
model.eval()
s = torch.randn(1,8,50).to(DEVICE)
for _ in range(10): _ = model(s)
if DEVICE.type=='mps': torch.mps.synchronize()
def cnn1d_single(x):
    xt = torch.from_numpy(x).float().to(DEVICE)
    with torch.no_grad(): out = model(xt)
    if DEVICE.type=='mps': torch.mps.synchronize()
    return out.argmax(1).cpu().numpy()
latency = measure_latency(cnn1d_single, X_train[:1], n_runs=500)
print_latency(latency, '1D-CNN')
print_comparison(zero_results, cal_results, name='1D-CNN (v6)')


Latency — 1D-CNN
  Mean:   1.46 ms
  Median: 1.27 ms
  P95:    1.80 ms
  <300ms: ✓

  1D-CNN (v6) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                58.62%       70.46%   +11.84%
S2                53.52%       76.57%   +23.05%
S3                53.98%       76.75%   +22.77%
S4                65.56%       81.37%   +15.80%
S5                80.29%       86.23% ✓   +5.94%
